# Random Guide-Selection Strategies

Compares different random strategies for selecting guide sets.
Each strategy gets its own run directory matched by name pattern.

### TODOs
- Add best single guide comparison (k=1 vs k>1)
- Revisit metrics — consider adding new ones

In [ ]:
import json

import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from helpers import find_latest_run, load_top_k

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 8)})

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# Add new strategies here: (display_name, run-directory pattern)
STRATEGIES = [
    ("random", "random"),
]

In [ ]:
# ── Load & parse ──────────────────────────────────────────────────────
all_rows = []
for name, pattern in STRATEGIES:
    run_dir = find_latest_run(pattern)
    print(f"{name}: {run_dir}")
    rows = load_top_k(run_dir, name)
    print(f"  {len(rows)} trial rows")
    all_rows.extend(rows)

df = pl.DataFrame(all_rows).with_columns(
    (pl.col("nodes") / pl.col("classes")).alias("nodes_per_class"),
)
k_values = sorted(df["k"].unique().to_list())
strat_names = [s[0] for s in STRATEGIES]

print(f"\nTotal rows: {len(df)}, k values: {k_values}")
df.head(5)

## Metrics vs k

In [ ]:
PALETTE = plt.cm.tab10.colors  # type: ignore[attr-defined]
MARKER_CYCLE = ["o", "s", "D", "^", "v", "P", "X", "*"]


def strat_style(i):
    return {
        "color": PALETTE[i % len(PALETTE)],
        "marker": MARKER_CYCLE[i % len(MARKER_CYCLE)],
    }


METRICS = ["iters", "nodes", "classes", "nodes_per_class", "total_time"]
n_metrics = len(METRICS)
fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5))

for metric, ax in zip(METRICS, axes):
    for i, strat in enumerate(strat_names):
        style = strat_style(i)
        agg = (
            df.filter(pl.col("strategy") == strat)
            .group_by("k")
            .agg(
                pl.col(metric).mean().alias("mean"),
                pl.col(metric).std().alias("std"),
            )
            .sort("k")
        )
        ks = agg["k"].to_numpy()
        means = agg["mean"].to_numpy()
        stds = agg["std"].fill_null(0).to_numpy()
        ax.plot(ks, means, label=strat, **style)
        ax.fill_between(
            ks, means - stds, means + stds, alpha=0.15, color=style["color"]
        )

    ax.set_xlabel("k (number of guides)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs k (mean \u00b1 std)")
    ax.legend()
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.set_xticks(k_values)

fig.tight_layout()
plt.show()

## Reachability rate vs k

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for i, strat in enumerate(strat_names):
    style = strat_style(i)
    reach = (
        df.filter(pl.col("strategy") == strat)
        .group_by("k")
        .agg(pl.col("iters").is_not_null().mean().alias("reach_rate"))
        .sort("k")
    )
    ax.plot(
        reach["k"].to_numpy(),
        reach["reach_rate"].to_numpy() * 100,
        label=strat,
        **style,
    )

ax.set_xlabel("k (number of guides)")
ax.set_ylabel("reachability (%)")
ax.set_title("Fraction of trials reaching the goal vs k")
ax.legend()
ax.set_xscale("log")
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set_xticks(k_values)
ax.set_ylim(-5, 105)
fig.tight_layout()
plt.show()

## Box plots: iterations by strategy at each k

In [ ]:
n_k = len(k_values)
fig, axes = plt.subplots(1, n_k, figsize=(4 * n_k, 5), squeeze=False, sharey=True)

for ki, k in enumerate(k_values):
    ax = axes[0][ki]
    box_data = []
    labels = []
    colors = []
    for i, strat in enumerate(strat_names):
        vals = (
            df.filter((pl.col("k") == k) & (pl.col("strategy") == strat))["iters"]
            .drop_nulls()
            .to_numpy()
        )
        if len(vals) > 0:
            box_data.append(vals)
            labels.append(strat)
            colors.append(PALETTE[i % len(PALETTE)])
    if box_data:
        bp = ax.boxplot(box_data, tick_labels=labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    ax.set_title(f"k = {k}")
    ax.set_ylabel("iters" if ki == 0 else "")

fig.suptitle("Iterations by strategy at each k", fontsize=13)
fig.tight_layout()
plt.show()

## Box plots: nodes by strategy at each k

In [ ]:
fig, axes = plt.subplots(1, n_k, figsize=(4 * n_k, 5), squeeze=False, sharey=True)

for ki, k in enumerate(k_values):
    ax = axes[0][ki]
    box_data = []
    labels = []
    colors = []
    for i, strat in enumerate(strat_names):
        vals = (
            df.filter((pl.col("k") == k) & (pl.col("strategy") == strat))["nodes"]
            .drop_nulls()
            .to_numpy()
        )
        if len(vals) > 0:
            box_data.append(vals)
            labels.append(strat)
            colors.append(PALETTE[i % len(PALETTE)])
    if box_data:
        bp = ax.boxplot(box_data, tick_labels=labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
    ax.set_title(f"k = {k}")
    ax.set_ylabel("nodes" if ki == 0 else "")

fig.suptitle("Nodes by strategy at each k", fontsize=13)
fig.tight_layout()
plt.show()

## Summary table

In [ ]:
summary = (
    df.group_by("strategy", "k")
    .agg(
        pl.col("iters").mean().round(2).alias("avg_iters"),
        pl.col("nodes").mean().round(0).alias("avg_nodes"),
        pl.col("classes").mean().round(0).alias("avg_classes"),
        pl.col("nodes_per_class").mean().round(2).alias("avg_nodes_per_class"),
        pl.col("total_time").mean().round(4).alias("avg_time_s"),
        pl.col("iters").is_not_null().mean().round(3).alias("reachability"),
        pl.col("iters").count().alias("n_trials"),
    )
    .sort("k", "strategy")
)
summary